In [1]:
# Standard library imports
import os
import sys

# Third-party imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Configure matplotlib
plt.style.use('fig.style')
figsize = (8,4)

# Add project root to path
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.append(project_root)

# Local imports
from src.get_hg_projection_matrix import get_hg_projection_matrix
from src.matrix_visualizer import visualize_matrix

from src import get_pairwise_model_features
from src import get_alphabet
from src.sort_features import sort_features

In [2]:
## Pairwise model

L = 10
alphabet = get_alphabet('protein')
alpha = len(alphabet)
num_to_show = 10
features = get_pairwise_model_features(L=L, alphabet=alphabet)
features = sort_features(features)
N = len(features)
print(f'pairwise:\n{features[:num_to_show]=}\n{features[-num_to_show:]=}\n')

pairwise:
features[:num_to_show]=['**********', 'A*********', 'C*********', 'D*********', 'E*********', 'F*********', 'G*********', 'H*********', 'I*********', 'K*********']
features[-num_to_show:]=['********MY', '********NY', '********PY', '********QY', '********RY', '********SY', '********TY', '********VY', '********WY', '********YY']



In [3]:
from src.augseq_features_to_petti_features import augseq_features_to_petti_features
petti_features = augseq_features_to_petti_features(features)
petti_features

[(0, (), ''),
 (1, (0,), 'A'),
 (1, (0,), 'C'),
 (1, (0,), 'D'),
 (1, (0,), 'E'),
 (1, (0,), 'F'),
 (1, (0,), 'G'),
 (1, (0,), 'H'),
 (1, (0,), 'I'),
 (1, (0,), 'K'),
 (1, (0,), 'L'),
 (1, (0,), 'M'),
 (1, (0,), 'N'),
 (1, (0,), 'P'),
 (1, (0,), 'Q'),
 (1, (0,), 'R'),
 (1, (0,), 'S'),
 (1, (0,), 'T'),
 (1, (0,), 'V'),
 (1, (0,), 'W'),
 (1, (0,), 'Y'),
 (1, (1,), 'A'),
 (1, (1,), 'C'),
 (1, (1,), 'D'),
 (1, (1,), 'E'),
 (1, (1,), 'F'),
 (1, (1,), 'G'),
 (1, (1,), 'H'),
 (1, (1,), 'I'),
 (1, (1,), 'K'),
 (1, (1,), 'L'),
 (1, (1,), 'M'),
 (1, (1,), 'N'),
 (1, (1,), 'P'),
 (1, (1,), 'Q'),
 (1, (1,), 'R'),
 (1, (1,), 'S'),
 (1, (1,), 'T'),
 (1, (1,), 'V'),
 (1, (1,), 'W'),
 (1, (1,), 'Y'),
 (1, (2,), 'A'),
 (1, (2,), 'C'),
 (1, (2,), 'D'),
 (1, (2,), 'E'),
 (1, (2,), 'F'),
 (1, (2,), 'G'),
 (1, (2,), 'H'),
 (1, (2,), 'I'),
 (1, (2,), 'K'),
 (1, (2,), 'L'),
 (1, (2,), 'M'),
 (1, (2,), 'N'),
 (1, (2,), 'P'),
 (1, (2,), 'Q'),
 (1, (2,), 'R'),
 (1, (2,), 'S'),
 (1, (2,), 'T'),
 (1, (2,), 'V'),


In [ ]:
# Create random theta
theta_series = pd.Series(index=features, data=np.random.randn(N))
theta_series

In [ ]:
from src.pairwise_theta_series_to_dict import pairwise_theta_series_to_dict
theta_dict = pairwise_theta_series_to_dict(theta_series, alphabet=alphabet, wildcard='*')
theta_dict

In [ ]:
# Fix gauge
from src.fix_gauge_pairwise_theta_dict import fix_gauge_pairwise_theta_dict
p_lc = (1/alpha)*np.ones((L,alpha))
fixed_theta_dict = fix_gauge_pairwise_theta_dict(theta_dict=theta_dict, p_lc=p_lc)
fixed_theta_dict

In [ ]:
from src.pairwise_theta_dict_to_series import pairwise_theta_dict_to_series
theta_fixed_series= pairwise_theta_dict_to_series(fixed_theta_dict)
theta_fixed_series

In [ ]:
from src.seq_embedder import SeqEmbedder
import random

# Get encoder; use to encode sequences
embedder = SeqEmbedder(features=features)

# Get projection matrix
from src.get_pairwise_zs_projection_matrix import get_pairwise_zs_projection_matrix
P = get_pairwise_zs_projection_matrix(L=L, alphabet=alphabet)

# Check that theta_fixed_series is close to projected theta_series
assert np.all(np.isclose(theta_fixed_series, P@theta_series))
print(f'{np.all(np.isclose(theta_fixed_series, P@theta_series))=}')

# Check that gauge matches for every sequence
num_seqs_to_test = 100
for seq_num in range(num_seqs_to_test):
    seq = ''.join(random.choices(alphabet, k=L))

    # Get embedded sequence
    x =embedder.embed(seq)

    # Compute function using the two vectors
    f = theta_series@x
    f_fixed = theta_fixed_series@x
    
    # Check that the two functions are close
    assert np.isclose(f, f_fixed), f'{f=}\n{f_fixed=}'
    
print(f'Tested {num_seqs_to_test} random sequences; all seqs passed.')

In [ ]:
import matplotlib.pyplot as plt
plt.scatter(theta_series, theta_fixed_series, s=1, alpha=1)
plt.xlabel('theta_series')
plt.ylabel('theta_fixed_series')

In [9]:
# Got the mavenn gauge fixing working for pairwise models. 
# Slowest step by far is translating theta_dict to theta_series. 
# Without this it seems pretty fast. 
# And I can probably speed this up by asigning values in the series with block asignments. 
# Actually yeah, I think this should speed things up a fair amount, maybe ~alpha**2 fold because that's how much the look ups will be reduced. 

# P = get_hg_projection_matrix(features=features, alphabet=alphabet, bg_type='uniform', wildcard_char='*', out_type='df')
# visualize_matrix(P.values, show_grid=False, figsize=figsize)

